## Notebook for testing our pipeline for the Generative Recommender

In [1]:
import sys
print(sys.executable)

/opt/anaconda3/envs/gen-rec/bin/python


In [2]:
# Imports here
from DataHandler import get_article_feature_string_list
from embedder import Embedder
from rqvae import RQVAE
from rqvae_train import rqvae_loss, train_rqvae_sanity_check, train_rqvae_full, load_trained_rqvae
import torch
import numpy as np
import pickle
import os
import time
import joblib
import torch, numpy as np, random
import pandas as pd
from transformer import recommended_next_sid, is_model_trained
from train_transformer import start_training
from transformers import BartTokenizer, BartForConditionalGeneration
from collaborative_filtering import compute_item_user_matrix, recommend_for_user
from evaluation import MAP_at_K

# This is ONLY for our tests! Do not use for our full model
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

### First, get the feature strings from DataHandler

In [3]:
feature_strings = get_article_feature_string_list()
# feature_strings is a list of lists, use list comprehension to make it a list of strings (first 1000 items)
feature_strings = [item[0] for item in feature_strings]
item_ids = [feature_string.split()[0] for feature_string in feature_strings]

print(f'Number of items: {len(feature_strings)}')
print(feature_strings[0])


Number of items: 105542
108775015 Solid Dark Black Strap top Jersey top with narrow shoulder straps. Vest top Garment Upper body Jersey Basic Ladieswear Ladieswear Womens Everyday Basics Jersey Basic


### Next, we use the embedder to get the embeddings that will serve as input for RQ-VAE

In [4]:
if os.path.exists('SBERT_embeddings_fulldata.npy'):
    embeddings = np.load('SBERT_embeddings_fulldata.npy')
    print(f'The embeddings have the shape: {embeddings.shape}')
else:
    embedder = Embedder("sbert")
    embeddings = embedder.encode(feature_strings) # embeddings has shape (n, d), where d = 384 for SBERT
    print(f'The embeddings have the shape: {embeddings.shape}')
    # print(embeddings[0])
    np.save('SBERT_embeddings_fulldata.npy', embeddings)

The embeddings have the shape: (105542, 384)


### Now, we can use the embeddings with RQ-VAE. The code loads hashmaps between item IDs and semantic IDs if they exist. If not, it uses the model to generate semantic IDs and creates (and saves) the hashmaps

In [5]:
# input_dim = embeddings.shape[1]
# latent_dim = 32
# hidden_dim = 256
# codebook_size = 512
# num_quantizers = 4

# TRAINING_MODE = 'None' # for loading

# set_seed(42)
# rqvae = RQVAE(input_dim=input_dim, latent_dim=latent_dim, hidden_dim=hidden_dim, codebook_size=512, num_quantizers=num_quantizers)

# # We need to train the model here, before retrieving semantic IDs! Omitting for now... 
# # trained_model = ...

# print('Generating semantic IDs')

# if isinstance(embeddings, np.ndarray):
#     embeddings = torch.from_numpy(embeddings).float()

# if TRAINING_MODE == 'sanity_check':
#     print("Running sanity check for RQ-VAE")
#     rqvae = train_rqvae_sanity_check(rqvae, embeddings)

# elif TRAINING_MODE == 'full_training':
#     print("Training RQ-VAE")
#     rqvae = train_rqvae_full(rqvae, embeddings, save_path='./models/rqvae')

# else:
#     rqvae = load_trained_rqvae(rqvae, 'quantizers/rqvae_training_results_20251101/models/rqvae_20251101_best.pth')

# semantic_ids = rqvae.encode_to_semantic_ids(embeddings)
# semantic_ids = semantic_ids.detach().numpy() # List of semantic IDs

# print(f'semantic_ids has shape: {semantic_ids.shape}')
# print('printing the first semantic ID (trained RQ-VAE)')
# print(semantic_ids[0])

item_2_semantic = {}
semantic_2_item = {}

if os.path.exists('quantizers/rqvae_training_results_20251127_132421/item_2_semantic_20251127_132421.pkl') and os.path.exists('quantizers/rqvae_training_results_20251127_132421/semantic_2_item_20251127_132421.pkl'):
    print('Loading existing hashmaps')

    with open('quantizers/rqvae_training_results_20251127_132421/item_2_semantic_20251127_132421.pkl', 'rb') as f:
        item_2_semantic = pickle.load(f)
    with open('quantizers/rqvae_training_results_20251127_132421/semantic_2_item_20251127_132421.pkl', 'rb') as f:
        semantic_2_item = pickle.load(f)
    print('Loaded the hashmaps')

else:
    for item_id, semantic_id in zip(item_ids, semantic_ids):
        semantic_tuple = tuple(semantic_id.astype(int))
        offset = 0
        semantic_tuple = (*semantic_tuple, offset)
        
        while semantic_tuple in semantic_2_item:
            offset += 1
            semantic_tuple = (*semantic_tuple[:-1], offset)
            
        item_2_semantic[item_id] = semantic_tuple
        semantic_2_item[semantic_tuple] = item_id

    print(f'Done creating hashmaps. Saving...')
    with open('item_2_semantic.pkl', 'wb') as f:
        pickle.dump(item_2_semantic, f)

    with open('semantic_2_item.pkl', 'wb') as f:
        pickle.dump(semantic_2_item, f)
    print(f'Saved hashmaps to files.')

Loading existing hashmaps
Loaded the hashmaps


### Converting transactions data (train, val and test) into: 
#### user_id: item_id1, ..., item_idn    --->   user_id: item_SID1, ..., item_SIDn

In [6]:
transactions_train = pd.read_pickle('transaction_list_train.pkl')
transactions_val = pd.read_pickle('transaction_list_val.pkl')
transactions_test = pd.read_pickle('transaction_list_test.pkl')

customer_transactions_train = {}
customer_transactions_val = {}
customer_transactions_test = {}

if os.path.exists('customer_transactions_train.pkl') and os.path.exists('customer_transactions_val.pkl') and os.path.exists('customer_transactions_test.pkl'):
    print('Loading existing customer_transactions_train and customer_transactions_val')

    with open('customer_transactions_train.pkl', 'rb') as f:
        customer_transactions_train = pickle.load(f)
    with open('customer_transactions_val.pkl', 'rb') as f:
        customer_transactions_val = pickle.load(f)
    with open('customer_transactions_test.pkl', 'rb') as f:
        customer_transactions_test = pickle.load(f)

    print('Loaded existing customer_transactions_train and customer_transactions_val')
else:

    for customer_id, group in transactions_train.groupby("customer_id"):
        item_ids = group["article_id"].tolist()[0]
        sid_list = []
        for item_id in item_ids:
            sid = item_2_semantic.get(str(item_id))
            if sid is not None:
                sid_list.append(sid)
            else:
                print(f"Warning: missing mapping for article_id {item_id}.")
        customer_transactions_train[customer_id] = sid_list
    
    for customer_id, group in transactions_val.groupby("customer_id"):
        item_ids = group["article_id"].tolist()[0]
        sid_list = []
        for item_id in item_ids:
            sid = item_2_semantic.get(str(item_id))
            if sid is not None:
                sid_list.append(sid)
            else:
                print(f"Warning: missing mapping for article_id {item_id}.")
        customer_transactions_val[customer_id] = sid_list
    
    for customer_id, group in transactions_test.groupby("customer_id"):
        item_ids = group["article_id"].tolist()[0]
        sid_list = []
        for item_id in item_ids:
            sid = item_2_semantic.get(str(item_id))
            if sid is not None:
                sid_list.append(sid)
            else:
                print(f"Warning: missing mapping for article_id {item_id}.")
        customer_transactions_test[customer_id] = sid_list
        
    print("customer_transactions_train, customer_transactions_val and customer_transactions_test have been created, saving...")
    with open("customer_transactions_train.pkl", "wb") as f:
        pickle.dump(customer_transactions_train, f)
    with open('customer_transactions_val.pkl', 'wb') as f:
        pickle.dump(customer_transactions_val, f)
    with open('customer_transactions_test.pkl', 'wb') as f:
        pickle.dump(customer_transactions_test, f)
    print("Saving done!")

# sanity check
first_customer = list(customer_transactions_test.keys())[0]
print(f"Customer_id: {first_customer}")
print(f"Train sequence => length: {len(customer_transactions_train[first_customer])} : {customer_transactions_train[first_customer]}")
print(f"Val sequence => length: {len(customer_transactions_val[first_customer])} : {customer_transactions_val[first_customer]}")
print(f"Test sequence => length: {len(customer_transactions_test[first_customer])} : {customer_transactions_test[first_customer]}")

Loading existing customer_transactions_train and customer_transactions_val
Loaded existing customer_transactions_train and customer_transactions_val
Customer_id: 00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657
Train sequence => length: 19 : [(np.int64(172), np.int64(76), np.int64(167), np.int64(214), np.int64(381), 0), (np.int64(42), np.int64(71), np.int64(151), np.int64(263), np.int64(104), 0), (np.int64(261), np.int64(156), np.int64(181), np.int64(120), np.int64(257), 0), (np.int64(58), np.int64(278), np.int64(446), np.int64(447), np.int64(434), 0), (np.int64(493), np.int64(145), np.int64(296), np.int64(73), np.int64(53), 0), (np.int64(493), np.int64(145), np.int64(296), np.int64(73), np.int64(53), 0), (np.int64(156), np.int64(129), np.int64(476), np.int64(386), np.int64(301), 0), (np.int64(144), np.int64(336), np.int64(80), np.int64(34), np.int64(51), 0), (np.int64(416), np.int64(294), np.int64(159), np.int64(6), np.int64(35), 0), (np.int64(493), np.int64(163), np.i

### Training the transformer or load it from the disk

In [7]:

if is_model_trained(): 
    print('The model is loaded...')
    model = BartForConditionalGeneration.from_pretrained('./bart-recommender/final_model')
    tokenizer = BartTokenizer.from_pretrained('./bart-recommender/final_model')
else: 
    print('There is no pretrained model, the model will be trained ...')
    start_training() # train the model

    

The model is loaded...


### Retrieving articles info for mapping

In [8]:
articles = pd.read_pickle('articles.pkl')
article_id = 108775044
row = articles.loc[articles['article_id'] == article_id]
product_name = row['prod_name']
product_name2 = row['prod_name'].iloc[0]
print(product_name)
print(product_name2)

1    Strap top
Name: prod_name, dtype: object
Strap top


### Inference: 

### Sanity check: GR-recommend SIDs for a given customer transaction sequence

In [9]:
last_10_customers = list(customer_transactions_val.keys())[-1:]
test_customers_sequences = {k: customer_transactions_val[k] for k in last_10_customers}

for customer_id, seqs in test_customers_sequences.items():
    test_seq = test_customers_sequences[customer_id]
    test_seq = [' '.join(tuple(str(x) for x in sid)) for sid in test_seq]
    print(f"Customer: {customer_id}")
    print(f"Input SIDs sequence: {test_seq}")
    recommended_sids = recommended_next_sid(test_seq, model, tokenizer, window_size=36, top_k=12)
    print(recommended_sids)
    filtered = []
    for sid in recommended_sids: # to filter non-existing keys
        key = tuple(sid)
        if not key:
            continue
        item = semantic_2_item.get(key)
        if item is None:
            continue
        filtered.append((sid, item))
    
    print("Recommended SIDs, generated by transformer: ")
    for sid, id in filtered:
        row = articles.loc[articles['article_id'].astype(str) == str(id)]
        prod_name = row['prod_name'].iloc[0]
        print(f'Rec. SID: {sid} --> article_id: {id} --> product_name: {prod_name}')

Customer: ffffd7744cebcf3aca44ae7049d2a94b87074c3d4ffe38b2236865d949d4df6a
Input SIDs sequence: ['474 404 371 8 389 0', '451 30 66 16 339 0', '317 218 374 169 70 0', '91 152 452 473 294 0', '20 78 371 152 46 0', '91 152 452 473 294 0']
[[56, 500, 371, 442, 204, 2], [56, 500, 371, 442, 204, 1], [56, 500, 371, 442, 204, 0], [56, 500, 71, 442, 204, 2], [314, 20, 250, 457, 431, 1], [56, 500, 71, 208, 389, 0], [56, 500, 71, 442, 204, 0], [56, 500, 71, 442, 204, 1], [56, 500, 371, 208, 389, 0], [56, 30, 471, 337, 392, 0], [56, 500, 371, 208, 403, 0], [56, 500, 371, 208, 257, 0]]
Recommended SIDs, generated by transformer: 
Rec. SID: [56, 500, 371, 442, 204, 2] --> article_id: 717490064 --> product_name: Cat Tee.
Rec. SID: [56, 500, 371, 442, 204, 1] --> article_id: 717490057 --> product_name: Cat Tee.
Rec. SID: [56, 500, 371, 442, 204, 0] --> article_id: 717490010 --> product_name: Cat Tee.
Rec. SID: [314, 20, 250, 457, 431, 1] --> article_id: 681180009 --> product_name: Panda skate dress j


# Evaluation Code Generative Recommendation (1% of test dataset)

In [10]:
model = BartForConditionalGeneration.from_pretrained('./bart-recommender/final_model')
tokenizer = BartTokenizer.from_pretrained('./bart-recommender/final_model')

transaction_list_eval = pd.read_pickle("customer_transactions_test.pkl") # evaluation on test dataset
items = list(transaction_list_eval.values())
fraction_01p = int(len(items) * 0.01)
transaction_list_eval_sample = items[:fraction_01p]

# Have a U x 1 np.array with labels
lab = [lst[-1] for (lst) in transaction_list_eval_sample]
lab = [' '.join(tuple(str(x) for x in sid)) for sid in lab]

# Recommend K products to U customers
start = time.time()
pred = []
for seq in transaction_list_eval_sample:
    seq_norm = [' '.join(map(str, sid)) for sid in seq]
    rec = recommended_next_sid(seq_norm, model, tokenizer, window_size=36, top_k=12)
    rec_str = [' '.join(map(str, sid)) for sid in rec]
    pred.append(rec_str)
elapsed_time = time.time() - start


# Evaluate using both pred and lab
map12 = MAP_at_K(pred, lab)
print(f"MAP@12 for GR is {map12}")
print(f"Total elapsed time for input size {len(transaction_list_eval_sample)}: {elapsed_time}")

MAP@12 for GR is 0.03187391867100872
Total elapsed time for input size 7629: 3690.2432491779327


# Evaluation Code Collaborative filtering (1% of test dataset)

In [11]:
if os.path.exists('CF_cashe.joblib'):
    unique_items, user2idx, interaction_matrix, item_similarity = joblib.load('CF_cashe.joblib')
else:
    unique_items, user2idx, interaction_matrix, item_similarity = compute_item_user_matrix()
    joblib.dump((unique_items,user2idx, interaction_matrix, item_similarity), "CF_cashe.joblib")

transaction_list_eval = pd.read_pickle("transaction_list_test.pkl")
fraction_01p = int(len(transaction_list_eval)*0.01)
transaction_list_eval01p = transaction_list_eval[:fraction_01p]
lab = [lst[-1] for lst in transaction_list_eval01p.iloc[:, 1]]
start = time.time()
pred = [recommend_for_user(customer_id, unique_items, user2idx, interaction_matrix, item_similarity, top_k=12)
             for customer_id, _ in transaction_list_eval01p.values] 
elapsed_time = time.time() - start 

map12 = MAP_at_K(pred, lab)
print(f"MAP@12 for CF is {map12}")
print(f"Total elapsed time for input size {len(transaction_list_eval01p)}: {elapsed_time}")

MAP@12 for CF is 0.024356271741250112
Total elapsed time for input size 7629: 5.1781229972839355
